## Exploratory data anaylsis of CyCIF dataset


In [ ]:
import os
import sys
import gc
import cv2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import tifffile
import xml.etree.ElementTree as ET

from skimage.transform import rescale
from IPython.display import display

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
sys.path.append('..')
import IO, zonation

### CyCIF multiplexed image registration

#### Full 3D sample

| Cycle | Opal 520 | Opal 690 | Opal 570 | Opal 780 | 
| --- | --- | --- | --- | --- |
| 1 | B-Catenin | ASS1 | GS | CYP3A4 |
| 2 | Pan-CK | CD31 | Collagen | NaN |
| 3 | CD45 | CD68 | Arg1 | Lyve1 |
| 4 | CD56 | Vimentin | PU1 | CD3 |

(1). Load Cy-IF images from google cloud bucket

In [ ]:
CREDENTIAL_PATH = "../data/azizilab-cell-segmentation-05f1a1125db2.json"
PROJECT_ID = 'azizilab-cell-segmentation'
BUCKET_ID = 'liver_3d'
HOME_PATH = 'CyIF/qupath/'

cyif_reader = IO.CyIFGcloudReader(CREDENTIAL_PATH,
                                  PROJECT_ID,
                                  BUCKET_ID,
                                  HOME_PATH,
                                  scale=1,
                                  sigma=10)


(2). Within-cycle registration

In [ ]:
out_path = '../data/cycif/raw/'

for slide_id in cyif_reader.slide_ids:
    annot_imgs = cyif_reader.load_imgs(slide_id,
                                       chans_to_ignore={'Opal 620'},
                                       verbose=True)    
    
    annot_imgs_warped = cyif_reader.register_cycles(annot_imgs, 
                                                    verbose=True)
    
    cyif_reader.save_annotated_imgs(annot_imgs_warped, 
                                    output_path=out_path)

    gc.collect()
    # del annot_imgs, annot_imgs_warped

### Intensity Adjustment

Try both heuristic & data-driven

In [ ]:
from skimage.exposure import adjust_gamma

(1). Heuristic min-max intensity & gamma adjustment

In [ ]:
data_path = '../data/cycif/raw'
out_path = '../data/cycif/adj'

if not os.path.exists(out_path):
    os.makedirs(out_path, exist_ok=True)

In [ ]:
# TODO: add util function for local `qptiff` & `ome.tiff` IO

CREDENTIAL_PATH = "../data/azizilab-cell-segmentation-05f1a1125db2.json"
PROJECT_ID = 'azizilab-cell-segmentation'
BUCKET_ID = 'liver_3d'
HOME_PATH = 'CyIF/qupath/'

cyif_reader = IO.CyIFGcloudReader(CREDENTIAL_PATH,
                                  PROJECT_ID,
                                  BUCKET_ID,
                                  HOME_PATH)

filenames = sorted([os.path.join(data_path, f)
                    for f in os.listdir(data_path)
                    if f[-7:] == 'ome.tif'])

chan_meta_df = pd.read_csv('../data/CyIF_antibody_ref_sheet.csv', index_col=[0])
display(chan_meta_df.head())

int_meta_df = pd.read_csv('../data/CyIF_intensity_ref_sheet.csv', index_col=[0, 1, 2, 3])
int_meta_df.dropna(axis=1, how='all', inplace=True)
int_meta_df = int_meta_df.applymap(lambda x: x if pd.isna(x) else min(x, 255.0))

display(int_meta_df.head())

In [ ]:
# TMP: Switch channel orders starting from Slide 2
# swap (ASS, GS), (Col I, CD31), (Arg1, CD68), (Vimentin, PU1)

# tmp_filenames = filenames[4:]
# tmp_chan_lists = []

# for f in tmp_filenames:
#     ifile = open(f, 'rb')
#     tmp_chan_lists.append(IO.load_ome_labels(ifile, f)) 

# tissue_ids = [f.rpartition('/')[-1]
#               for f in tmp_filenames]

# # Swap orders (likely due to labeling issues btw Opal_570 & Opal_680)
# for i in range(len(tmp_chan_lists)):
#     idx1, idx2 = tmp_chan_lists[i].index('GS 647'), tmp_chan_lists[i].index('ASS1 PE')
#     tmp_chan_lists[i][idx1], tmp_chan_lists[i][idx2] = tmp_chan_lists[i][idx2], tmp_chan_lists[i][idx1]
    
#     idx1, idx2 = tmp_chan_lists[i].index('Col I'), tmp_chan_lists[i].index('CD31')
#     tmp_chan_lists[i][idx1], tmp_chan_lists[i][idx2] = tmp_chan_lists[i][idx2], tmp_chan_lists[i][idx1]

#     idx1, idx2 = tmp_chan_lists[i].index('Arg1'), tmp_chan_lists[i].index('CD68')
#     tmp_chan_lists[i][idx1], tmp_chan_lists[i][idx2] = tmp_chan_lists[i][idx2], tmp_chan_lists[i][idx1]

#     idx1, idx2 = tmp_chan_lists[i].index('PU1'), tmp_chan_lists[i].index('Vimentin')
#     tmp_chan_lists[i][idx1], tmp_chan_lists[i][idx2] = tmp_chan_lists[i][idx2], tmp_chan_lists[i][idx1]
    
# del i, idx1, idx2

# # Annotated imgs
# annot_imgs = {}
# for f, tid, labels in zip(tmp_filenames, tissue_ids, tmp_chan_lists):
#     img = tifffile.imread(f)
#     annot_img = {c: img_chan 
#                  for (c, img_chan) in zip(labels, img)}
    
#     annot_imgs[tid] = annot_img

# del f, tid, ifile, labels

# cyif_reader.save_annotated_imgs(annot_imgs, output_path=out_path, verbose=True)


In [ ]:
def adj_intensity(fname, chan_meta, int_meta, verbose=True):
    if verbose:
        print('Adjusting intensities for img {}...'.format(
            fname.rpartition('/')[-1]))

    # Load annotated image from '.tif'
    img = tifffile.imread(fname)
    ifile = open(fname, 'rb')
    labels = IO.load_ome_labels(ifile, fname)
    annot_img = {lbl: img_chan
                 for (lbl, img_chan) in zip(labels, img)}
    
    adjusted_img = {}
    adjusted_img['DAPI'] = annot_img['DAPI']

    # Adjust intensities
    cycles = chan_meta.index
    channels = chan_meta.columns

    for cycle in cycles:
        for channel in channels:
            lbl = chan_meta.loc[cycle, channel]
            if pd.isna(lbl) or lbl not in annot_img.keys():
                continue

            if verbose:
                print('\tChannel: {}'.format(lbl))

            # Retrieve heursitic target intensities
            mask_row = int_meta.index.get_level_values('Round Number') == cycle
            mask_col = ['Min-Display_'+channel, 'Max-Display_'+channel, 'Gamma']
            imin, imax, gamma = int_meta.loc[mask_row, mask_col].values.flatten()

            X = annot_img[lbl] # intensity of the given image chan
            X = (X - imin) / (imax - imin)
            X[X < 0] = 0
            X[X > 1] = 1
            X = adjust_gamma(X, gamma)

            # Rescale to [0, 255]
            adjusted_img[lbl] = np.round(255*X).astype(np.uint8)

    # Append AF channels
    for lbl in labels:
        if 'AF' in lbl:
            adjusted_img[lbl] = annot_img[lbl]

    return adjusted_img


In [ ]:
adjusted_imgs = {}

for filename in filenames:
    key = filename.rpartition('/')[-1]
    tissue_id = int(filenames[-1][-10:-8]) + 1
    int_meta = int_meta_df.loc[int_meta_df.index.get_level_values('Tissue') == tissue_id]
    adjusted_img = adj_intensity(filename, chan_meta_df, int_meta) 
    adjusted_imgs[key] = adjusted_img

    del adjusted_img
    gc.collect()

del filename, key, tissue_id

In [ ]:
cyif_reader.save_annotated_imgs(adjusted_imgs, output_path=out_path)

### 2D/3D zonation

- TODO: **Figure out misalignment issues**, currently only works for Slide 1 & 3
- Selected zonation markers: `GS 647`, `CYP3A4`, `ASS1 PE` (Round 1), `Col I`, `Pan CK` (Round 2)
- Try construct 3D graphs

In [ ]:
from skimage.filters import gaussian, sobel, threshold_otsu, threshold_minimum
from skimage.exposure import rescale_intensity
from skimage.registration import optical_flow_ilk
from scipy import ndimage as ndi

In [ ]:
from importlib import reload
reload(IO)

In [ ]:
def remove_holes(roi, min_area):
    """
    Remove holes & FP lslands in binary ROI mask
    """
    roi_filtered = roi.copy().astype(np.uint8)
    roi_labeled, n_features = ndi.label(roi)
    
    for i in range(1, n_features+1):
        if (roi_labeled == i).sum() < min_area:
            roi_filtered[roi_labeled == i] = 0
            
    return ndi.binary_fill_holes(roi_filtered).astype(np.uint8)


def get_roi_mask(dapi, sigma=10, min_area=None):
    """
    Compute slice foreground / background mask
    """
    if min_area is None:
        # Defaulting min_area as 1/4 of the whole FOV area
        min_area = dapi.shape[0]//2*dapi.shape[1]//2
        
    dapi_smoothed = rescale_intensity(gaussian(dapi, sigma=10), out_range=(0, 1))
    threshold = threshold_otsu(dapi_smoothed)
    return remove_holes(dapi_smoothed > threshold, min_area=min_area)
    

Load aligned CyIF tiffs:

In [ ]:
data_path = '../data/cycif/aligned/registered_slides/'
cyif_imgs, filenames = IO.load_annot_tiffs(data_path, ext='ome.tiff')

# TODO: rerun on full z-slices after solving the misalignment issues
cyif_imgs, filenames = cyif_imgs[:7], filenames[:7]
filenames

In [ ]:
chan_labels = cyif_imgs[0].keys()
print(list(chan_labels))

#### Examine continuity of zonation markers:

In [ ]:
# Compute ROI (foreground) mask  + AF mask for each z-slice
# roi_masks = [get_roi_mask(img['DAPI'])
#              for img in cyif_imgs]

af1_masks = [(img['Sample AF_01'] > 40).astype(np.uint8)
             for img in cyif_imgs]
af2_masks = [(img['Sample AF_02'] > 40).astype(np.uint8)
             for img in cyif_imgs]

In [ ]:
# Try subtracting w/ AF within ROI foregrond regions
# Normalize each channel to [0, 1]

zone_labels = ['GS 647', 'CYP3A4', 'ASS1 PE', 'Col I', 'Pan CK']
zonation_chans = {}

for label in zone_labels:
    chans = []
    for i, (img, roi) in enumerate(zip(cyif_imgs, roi_masks)):
        af = af1_masks[i] if label in zone_labels[:3] else af2_masks[i]
        
        chan = rescale_intensity(img[label], out_range=(0, 1)) - af
        chan[chan < 0] = 0
        chan[roi == 0] = 0
        chans.append(chan)

        # chan = rescale_intensity(img[label].copy(), out_range=(0, 1))
        # chan[roi == 0] = 0
        # chans.append(chan)
        
    zonation_chans[label] = np.array(chans)

del label, chans, img, roi, chan, af
# del img, roi, chan

In [ ]:
def disp_chans(img, title=None, ncols=4):
    depth = len(img)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(img[idx])
            idx += 1
            
    plt.tight_layout()
    plt.suptitle(title, y=1.01)
    plt.show()
    return None

In [ ]:
# Denoise w/ binarized AF
for label in zone_labels:
    disp_chans(zonation_chans[label], title=label)
del label

Signals of `GS`, `CYP` & partial `ASS` looks Okay. `Collagen` & `Pan-CK` seems X usable. Quantify slice-wise transitions:

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3))
ax1.imshow(zonation_chans['GS 647'].max(0))
ax1.set_title('GS')
ax2.imshow(zonation_chans['CYP3A4'].max(0))
ax2.set_title('CYP')
ax3.imshow(zonation_chans['ASS1 PE'].max(0))
ax3.set_title('ASS')

plt.suptitle("Maximum Intensity Projection on Z-slices", y=1.01)
plt.tight_layout()
plt.show()

**Proof-of-concept** experiments: cross-slice registration quality

In [ ]:
# Save sample overlayed zonation markers
for label, chan in zonation_chans.items():
    tifffile.imwrite('../data/cycif/backup/CyIF_{}_slices.tif'.format(label), chan, metadata={'axes': 'CYX'})

In [ ]:
def disp_optical_flow(z1, z2, title=None):
    # Reference: 
    # https://scikit-image.org/docs/stable/auto_examples/registration/plot_opticalflow.html
    v, u = optical_flow_ilk(z1, z2)
    norm = np.sqrt(u** + v**2)
    nvec = 20
    nl, nc = z1.shape
    step = max(nl//nvec, nc//nvec)
    
    y, x = np.mgrid[:nl:step, :nc:step]
    u_ = u[::step, ::step]
    v_ = v[::step, ::step]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 3))
    ax1.imshow(z1)
    ax1.set_title('Slice z')

    ax2.imshow(z2)
    ax2.set_title('Slice z+1')
    
    ax3.imshow(norm)
    ax3.quiver(x, y, u_, v_, color='r', units='dots',
               angles='xy', scale=0.8, scale_units='xy', lw=2)
    ax3.set_title("Optical flow vector field")

    plt.tight_layout()
    plt.suptitle(title, y=1.1)
    plt.show()

In [ ]:
for label in zone_labels[:3]:
    print('Channel', label)
    print('==='*5)
    for i in range(len(zonation_chans[label])-1):
        z1, z2 = zonation_chans[label][i], zonation_chans[label][i+1]
        disp_optical_flow(z1, z2, title='Transition of {0} slices({1} vs. {2})'.format(label, i, i+1))   
    print('\n\n\n')

del z1, z2

#### 3D heat diffusion 
- Trajectory origin: `ASS` for now; destination `GS`
- 6-connected components as the graph unit
- **Todo:** add anisotropy scaling along Z-axis

In [ ]:
import time
import networkx as nx
from scipy.sparse import linalg
from skimage.segmentation import find_boundaries
from skimage.morphology import binary_dilation


In [ ]:
def create_graph(roi):
    """
    Convert combinatory graph w/ 4-connected compoments:
    adj(x) = {(i+1, j, k), (i-1, j, k),
              (i, j+1, k), (i, j-1, k),
              (i, j, k+1), (i, j, k-1)}
    from corresponding ROI pixels from the input image
    
    Code Referenced from: https://stackoverflow.com/questions/63653267/how-to-create-a-graph-with-an-images-pixel
    """
    # Horizontal edge (X)
    hx, hy, hz =  np.nonzero(np.logical_and(roi[1:] == 1, roi[:-1] == 1))  # horizontal edge start positions
    h_units = np.array([hx, hy, hz]).T
    h_starts = [tuple(n) for n in h_units]
    h_ends = [tuple(n) for n in h_units + (1, 0, 0)] 
    horizontal_edges = zip(h_starts, h_ends)

    # Vertical edge (Y)
    vx, vy, vz = np.nonzero(np.logical_and(roi[:,1:,:] == 1, roi[:,:-1,:] == 1))  # vertical edge start positions
    v_units = np.array([vx, vy, vz]).T
    v_starts = [tuple(n) for n in v_units]
    v_ends = [tuple(n) for n in v_units + (0, 1, 0)] 
    vertical_edges = zip(v_starts, v_ends)

    # In-plane edge (Z)
    ux, uy, uz = np.nonzero(np.logical_and(roi[:,:,1:] == 1, roi[:,:,:-1] == 1))
    u_units = np.array([ux, uy, uz]).T
    u_starts = [tuple(n) for n in u_units]
    u_ends = [tuple(n) for n in u_units + (0, 0, 1)]
    inplane_edges = zip(u_starts, u_ends)

    # Create graph
    G = nx.Graph()
    G.add_edges_from(horizontal_edges)
    G.add_edges_from(vertical_edges)
    G.add_edges_from(inplane_edges)
    
    return G


def add_graph_props(G, cv_nodes, pv_nodes, depth):
    """
    Assign init. temp. & ROI boundary as graph properties
    """
    for n in G:
        if n in cv_nodes:
            G.nodes[n]['t'] = 1
            G.nodes[n]['bound'] = True
        elif n in pv_nodes:
            G.nodes[n]['t'] = -1
            G.nodes[n]['bound'] = True
        else:
            G.nodes[n]['t'] = 0
            # Only define ROI bound along XY plane
            if G.degree[n] < 6 and 0 < n[0] < depth-1: # 3D
                G.nodes[n]['bound'] = True
    return None


def compute_interior_temp(G, debug=False):
    """
    Compute temperature of "interior" nodes based on Harmonic interpolation solution (Grady & Schwartz, 2003)
    """
    # Constructed permuted Laplacian Matrix L => {L_b, L_i, R, R^T}
    bound_nodes = [
        n for n, v in G.nodes.items()
        if 'bound' in v
    ]

    interior_nodes = [
        n for n, v in G.nodes.items()
        if 'bound' not in v
    ]

    n_bound = len(bound_nodes)
    perm_node_orders = bound_nodes + interior_nodes
    if debug:
        assert len(G) == len(perm_node_orders)

    L = nx.laplacian_matrix(G, nodelist=perm_node_orders)
    L_i = L[n_bound:, n_bound:]
    R = L[:n_bound, n_bound:]

    # Validate permuted nodes' in-degree have the correct order [d(bound), d(interior)]
    if debug:
        diag = np.diag(L)
        for i, n in enumerate(perm_node_orders):
            assert G.degree[n] == diag[i]

    # Compute interior temperature u(i) from L & u(b): 
    # Sol:  L_i @ u(i) = -R^T @ u(b)
    u_b = np.asarray([G.nodes[n]['t'] for n in bound_nodes])
    u_i = linalg.cg(A=L_i, b=-R.T@u_b) 

    if isinstance(u_i, tuple):
        u_i = u_i[0]
    
    return u_i, tuple(np.array(interior_nodes).T)  # 2 x N tuple


def assign_diffusion_temp(
    u_i, 
    interior_coords,
    cv_coords,
    pv_coords, 
    shape
):
    """
    Assign steady-state sol. of the diffused pixel values back to the image
    """
    assert len(interior_coords[0]) == len(u_i), 'Different coords & temperature lengths'
    u = np.zeros(shape, dtype=np.float64)
    u[interior_coords] = u_i
    u[cv_coords] = 1
    u[pv_coords] = -1
    return u


def discretize_temp(u, roi, cv_coords, n_layers=10, return_border=False, verbose=False):
    """
    Create discretized 1-indexed bins (1,2,...,n) as the zonation priors 
    from diffused gradient temperature `u`, keep CV & PV regions off from 
    `roi` as the min (PV) / max (CV) zones
    """
    assert n_layers > 3, "Invalid `n_layers`, please assign # lobule layers > 3"
    
    coords = np.nonzero(roi)
    coords_to_rm = np.nonzero(1-roi)
    qs = np.quantile(u[coords], np.linspace(0, 1, n_layers-1))

    if verbose:
        print('Quantile:', qs)
        
    zone = np.zeros_like(u, dtype=np.int32)
    for i, q in enumerate(qs[:-1]):
        zone[u >= q] = i+1
    zone[coords_to_rm] = 0

    zone[cv_coords] = zone.max() + 1
    zone += 1

    # Assign 1-pixel width border btw zones
    if return_border:
        border = find_boundaries(zone)
        zone[border] = 0

    return zone

**(1)**. Construct 3D ROI & CV / PV tentative mask

**TODO**: expand # Z-slices & increase weights along in-plane (Z-axis) edges for anisotropy

In [ ]:
# CV / PV mask w/ otsu threshold

# roi = np.array(roi_masks)
# cv_mask = np.zeros_like(roi, dtype=np.int32)
# for i, chan in enumerate(zonation_chans['GS 647']):
#     threshold = threshold_otsu(chan)
#     cv_mask[i] = (chan > threshold).astype(np.int32)
# del chan, threshold

# pv_mask = np.zeros_like(roi, dtype=np.int32)
# for i, chan in enumerate(zonation_chans['ASS1 PE']):
#     threshold = threshold_otsu(chan)
#     pv_mask[i] = (chan > threshold).astype(np.int32)
# del chan, threshold

# mask = np.zeros_like(roi, dtype=np.int32)
# mask[np.logical_and(pv_mask == 1, cv_mask == 0)] = -1
# mask[np.logical_and(pv_mask == 0, cv_mask == 1)] = 1

# Dilate on mask
# for i, mask_slc in enumerate(mask):
#     cv_mask = binary_dilation(mask_slc == 1, np.ones((5, 5))).astype(np.uint8)
#     pv_mask = binary_dilation(mask_slc == -1, np.ones((5, 5))).astype(np.uint8)

#     mask[i][pv_mask == 1] = -1
#     mask[i][cv_mask == 1] = 1

# del cv_mask, pv_mask

# for i, (roi_slc, mask_slc) in enumerate(zip(roi, mask)):
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
#     ax1.imshow(roi_slc)
#     ax1.set_title('Tissue ROI (z={})'.format(i))
#     ax1.axis('off')
    
#     ax2.imshow(mask_slc, cmap='coolwarm')
#     ax2.set_title('CV/PV mask (z={})'.format(i))
#     ax2.axis('off')
    
#     plt.tight_layout()
#     plt.show()

# del roi_slc, mask_slc

# # Save current ROI & CV/PV mask
# tifffile.imwrite('../data/cycif/backup/CyIF_ROI.tif', roi)
# tifffile.imwrite('../data/cycif/backup/CyIF_zone_masks.tif', mask)

**(2)** Construct 3D graph & run diffusion inference

In [ ]:
# 3D heat diffusion model

# Load computed roi & mask
# DEBUG: 3D diffusion's correction w/ Z-slices 8-11
roi = tifffile.imread('../data/cycif/backup/CyIF_ROI.tif')[-4:]
mask = tifffile.imread('../data/cycif/backup/CyIF_zone_masks.tif')[-4:]


G = create_graph(roi=roi)
add_graph_props(G, cv_indices, pv_indices, depth=roi.shape[0])
u_i, interior_nodes = compute_interior_temp(G)
u = assign_diffusion_temp(u_i, 
                          interior_nodes,
                          cv_coords=np.where(mask == 1),
                          pv_coords=np.where(mask == -1),
                          shape=mask.shape)

In [ ]:
t0 = time.perf_counter()

model = zonation.HeatDiffusion(vein_prior=mask,
                               roi=roi,
                               ndim=3)
U_i, interior_nodes = model.get_interior_U()
U = model.infer_zone_dynamics()
# lobule_layers = model.infer_zones()

t1 = time.perf_counter()
print('Heat diffusion for dim {0} x {1} x {2} takes {2}s'.format(mask.shape[0], mask.shape[1], mask.shape[2], t1-t0))
del U_i, interior_nodes


In [ ]:
for i, u_slc in enumerate(U):
    plt.figure()
    plt.imshow(u_slc, cmap='coolwarm')
    plt.title('Diffused dynamics (z={})'.format(i))
    plt.show()

In [ ]:
# Save inferred heat dynamics
tifffile.imwrite('../data/cycif/backup/CyIF_zone_dynamics.ome.tif', u, metadata={'axes': 'ZYX'})

---